# PKG Contagion Phase 1 — shock event study

**PNC Treasury Management · DSI · Payment Knowledge Graph (PKG)**

What happens to a TM organisation after a standing incoming payment stops, is
skipped or comes in short? Each shocked customer is compared with matched
unshocked customers, 6 months before to 6 months after the shock.

**This version (v1) runs on tables whose columns are confirmed:**
`pkg_cust_pair_month_v3` (shocks), `pkg_cust_metrics_monthly_v3` (outcomes),
`pkg_cpty_metrics_monthly` (hub flag). Shocks are therefore at **monthly grain**:
stop, skip, short. Delays inside a month need the periodicity table and are not in v1.

**Deposits, PxV and credit** plug in through `EXTRA_OUTCOMES` in CONFIG. They are
left empty on purpose: §0 prints the columns of each of those tables so the names
can be filled in without guessing. Until then those families are skipped, nothing breaks.

| § | What it does | Output |
|---|---|---|
| 0 | Inputs: required columns, months, optional tables | one table |
| 1 | Population: TM organisations, standing streams, baseline inflow | one table |
| 2 | Stream events and customer shocks | one table |
| 3 | Pre-flight counts, gates, headline choice | one table |
| 4 | Matched controls and the event panel | one table |
| 5 | Effects: event time, dose response, cuts, pre-trend check | tables + figures on disk |
| 6 | Second hop: do the shocked customer's own payees get hit? | table + figure on disk |
| 7 | Case panels | figures on disk |
| 8 | Answer sheet, gates, files written | one table |

Tables are shown inline (pandas, one output per cell). **Figures are saved to disk, never shown inline.**
Nothing is written to cdi_tm. Work files go to the personal HDFS folder, results to `OUT_DIR`.

In [ ]:
# =============================================================================
# SETUP — environment shims, CONFIG, Spark session, helpers
# =============================================================================
import os, sys, time, json, math, hashlib, warnings, threading, contextlib
import numpy as np
import pandas as pd

# --- environment compat (must run before any toPandas / createDataFrame) ------
for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp),
               ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

import matplotlib
matplotlib.use("Agg")                       # figures go to disk, never inline
import matplotlib.pyplot as plt
import matplotlib.ticker

from pyspark.sql import SparkSession, functions as F, Window as W

DB = "bdahd01p_dlcdi1_cdi_tm"

CONFIG = dict(
    # --- inputs (column names confirmed from the customer-metrics v3 build) ---
    MET_TABLE  = f"{DB}.pkg_cust_metrics_monthly_v3",
    PAIR_TABLE = f"{DB}.pkg_cust_pair_month_v3",
    CPTY_TABLE = f"{DB}.pkg_cpty_metrics_monthly",     # ent_id, month, hub_tier
    USE_HUB    = True,                                 # False = skip the hub flag

    # --- optional outcome families: fill the None values after reading §0 ----
    # id_kind is "customer_pwr_id" or "mdm_id". month_col must hold 'YYYY-MM'
    # strings or a date. measures: {output name: (source column, kind)} with
    # kind "index" (amounts, balances) or "diff" (rates, PD, flags).
    EXTRA_OUTCOMES = [
        dict(name="deposits", table=f"{DB}.pkg_dep_cust_month", id_col=None,
             id_kind="customer_pwr_id", month_col=None,
             measures={"dep_avg_bal": (None, "index")}),
        dict(name="pxv", table=f"{DB}.pkg_tm_cust_month", id_col=None,
             id_kind="customer_pwr_id", month_col=None,
             measures={"pxv": (None, "index")}),
        dict(name="credit", table="clhhd01p_clh.cdm_customer_fact_v", id_col=None,
             id_kind="customer_pwr_id", month_col=None,
             measures={"pd": (None, "diff"), "dhe": (None, "index")}),
    ],

    # --- window ---------------------------------------------------------------
    MONTH_MIN = "2024-01",
    MONTH_MAX = None,            # None = last month present in both inputs

    # --- run control ----------------------------------------------------------
    RUN_STAGE = "full",          # "preflight" stops after §3
    REBUILD   = False,           # True = ignore the work files of this configuration
    WORK_DIR  = "hdfs:///user/pk36814/pkg_contagion_p1/work",
    OUT_DIR   = "../metrics/contagion/p1",
    HDFS_REPLICATION   = 2,
    SHUFFLE_PARTITIONS = 800,

    # --- population -----------------------------------------------------------
    ORG_PARTY_TYPES = ["O"],     # organisations
    BASE_MONTHS     = 6,         # baseline = the 6 months before the event month
    BASE_IN_MIN     = 10_000.0,  # minimum average monthly incoming dollars at baseline

    # --- stream events (monthly grain) ----------------------------------------
    STANDING_MIN = 4,            # present in >= 4 of the last 6 months, incl. the last one
    STOP_MONTHS  = 2,            # missing 2+ months in a row = stop; 1 = skip
    SHORT_DROP   = 0.25,         # amount more than 25% below expected = short

    # --- customer shock -------------------------------------------------------
    TIER_EDGES    = [0.10, 0.20, 0.50],   # shock size tiers; first edge = minimum kept
    HEADLINE_MIN  = 0.20,
    CLEAN_MONTHS  = 6,           # event = first shock after 6 months without one
    MIN_FOLLOW    = 3,           # event months end 3 months before the last month
    PAYERWIDE_SHARE      = 0.50, # payer broke >= 50% of its standing payees that month
    PAYERWIDE_MIN_PAYEES = 5,
    HUB_TIER      = 2,           # hub_tier >= 2 = processor-like payer

    # --- gates (spec defaults) ------------------------------------------------
    GATE_MIN_EVENTS      = 1000, # fewer at 20%+ -> headline drops to 10%
    GATE_MIN_PAYERWIDE   = 0.10, # payer-wide share below this -> headline = all shocks

    # --- event study ----------------------------------------------------------
    PRE_MONTHS  = 6,
    POST_MONTHS = 6,
    N_CONTROLS  = 3,
    CONTROL_CLEAN_AFTER = 0,     # controls are shock-free for 6 months before and in the event month.
                                 # 0 = free to be shocked later, like any customer (unbiased);
                                 # 6 = also shock-free for 6 months after (overstates effects)
    PANEL_MAX_EVENTS = 30_000,   # events carried into the panel (deterministic sample above this)
    EFFECT_T    = 3,             # month used for dose response and cuts
    BOOT_REPS   = 200,
    INDEX_CAP   = 1000.0,        # index values are capped at 10x baseline
    SEED        = 20261113,

    # --- cases ----------------------------------------------------------------
    CASE_TOP_N  = 10,
    CASE_NAME_PATTERNS = ["UNITED CONSUMER FINANCIAL"],
)

# Outcomes read from the customer metrics. kind: index = 100 x value / own baseline;
# diff = change from own baseline; scaled = value / baseline inflow, change from baseline.
OUTCOMES = {
    "amt_all_in":               ("Incoming dollars",                         "index"),
    "amt_all_out":              ("Outgoing dollars",                         "index"),
    "ntxn_all_out":             ("Outgoing payment count",                   "index"),
    "active_days_out":          ("Days with outgoing payments",              "index"),
    "ncpty_ext_out":            ("External payees paid",                     "index"),
    "rec_standing_out":         ("Standing payees",                          "index"),
    "rec_broken_out":           ("Standing payees not paid (count)",         "diff"),
    "rec_broken_share_out":     ("Standing payees not paid (share, pp)",     "diff"),
    "rec_broken_amt_share_out": ("Standing payee dollars not paid (share, pp)", "diff"),
    "net_flow_all":             ("Net flow / baseline inflow (pp)",          "scaled"),
}
SHARE_COLS = {"rec_broken_share_out", "rec_broken_amt_share_out"}   # shown x100

MET_REQUIRED  = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                 "naics2", "is_tm", "pwr_shared", "month", "rec_standing_in"] + list(OUTCOMES)
PAIR_REQUIRED = ["cust", "dir", "scope", "other", "amount", "month"]
CPTY_REQUIRED = ["ent_id", "month", "hub_tier"]

# --- Spark session (ACC code-review limits: 4g / 4g / 1g, 512 executors) --------
spark = (SparkSession.builder.appName("pkg_contagion_p1")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.yarn.am.memory", "4g")
         .config("spark.yarn.am.memoryOverhead", "1g")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.kryoserializer.buffer.max", "128m")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .config("spark.hadoop.dfs.replication", str(CONFIG["HDFS_REPLICATION"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:    # cluster noise: Atlas lineage hook, Kafka, netty (harmless to data)
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
               "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
               "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

# --- display: one HTML output per cell; progress line above the tables ---------
try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False


def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def __init__(self):
        self.lines, self.tables, self.h = [], [], None

    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=60))

    def _draw(self):
        if self.h is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#52514e;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()
TIMINGS = []


@contextlib.contextmanager
def step(name):
    """Timed step with a live line: elapsed minutes and Spark tasks of the running stage."""
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        TIMINGS.append((name, round(mins, 2)))
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


GATES = []


def gate(name, ok, detail, action=""):
    GATES.append(dict(gate=name, result="PASS" if ok else "FAIL", detail=detail, action=action))


def gates_frame():
    return pd.DataFrame(GATES, columns=["gate", "result", "detail", "action"])


# --- months ---------------------------------------------------------------------
_BASE = int(CONFIG["MONTH_MIN"][:4]) * 12 + int(CONFIG["MONTH_MIN"][5:7])


def mi_col(c):
    """Integer month index, 0 = MONTH_MIN. Gaps are safe with rangeBetween windows."""
    c = F.col(c) if isinstance(c, str) else c
    return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int") - F.lit(_BASE)).cast("int")


def mi2m(i):
    k = int(i) + _BASE
    return f"{(k - 1) // 12:04d}-{(k - 1) % 12 + 1:02d}"


def table_months(tbl):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().iloc[:, 0]
        m = sorted({x.split("month=")[1].split("/")[0] for x in p if "month=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(r[0] for r in spark.table(tbl).select("month").distinct().collect())


# --- work files: one folder per configuration, so a changed threshold never reuses old files
_SIG_KEYS = ["MET_TABLE", "PAIR_TABLE", "CPTY_TABLE", "USE_HUB", "MONTH_MIN", "ORG_PARTY_TYPES",
             "BASE_MONTHS", "BASE_IN_MIN", "STANDING_MIN", "STOP_MONTHS", "SHORT_DROP", "TIER_EDGES",
             "CLEAN_MONTHS", "PAYERWIDE_SHARE", "PAYERWIDE_MIN_PAYEES", "HUB_TIER"]


def _work(name):
    return f"{CONFIG['WORK_DIR']}/sig={SIG}/{name}"


def cache_parquet(sdf, name):
    """Write once, read back. Later runs with the same configuration skip the build."""
    path = _work(name)
    if not CONFIG["REBUILD"]:
        try:
            out = spark.read.parquet(path)
            out.limit(1).count()
            CELL.log(f"reused work file: {name}")
            return out
        except Exception:
            pass
    sdf.write.mode("overwrite").parquet(path)
    return spark.read.parquet(path)


# --- figures: saved to disk, indexed, never shown inline --------------------------
OUT_DIR = CONFIG["OUT_DIR"]
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)
FIG_INDEX = []

INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
C_SHOCK, C_CTRL = "#2a78d6", "#8a8985"          # shocked = blue, control = neutral gray
C_IN, C_OUT, C_MARK = "#008300", "#e34948", "#eb6834"   # in green, out red (team chart rule)
TYPE_COLORS = {"stop": "#2a78d6", "skip": "#eb6834", "short": "#1baf7a", "open": "#eda100"}

plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": "#b8b7b0", "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 2.0, "figure.dpi": 110,
})


def savefig(fig, name, section, caption):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX.append(dict(file=f"{name}.png", section=section, caption=caption))
    return path


def save_plotly(fig, name, section, caption):
    """Standalone HTML copy (opens in the browser; inline Plotly is empty in this JupyterLab)."""
    try:
        fig.write_html(os.path.join(FIG_DIR, f"{name}.html"), include_plotlyjs=True, full_html=True)
        FIG_INDEX.append(dict(file=f"{name}.html", section=section, caption=caption + " (interactive)"))
    except Exception as ex:
        CELL.log(f"interactive copy of {name} skipped: {type(ex).__name__}")


def usd(v):
    return "" if v is None or (isinstance(v, float) and math.isnan(v)) else f"${v:,.0f}"


ANSWERS = []


def ans(q, a):
    ANSWERS.append(dict(question=q, answer=a))

## §0 · Inputs

Confirms every column this notebook depends on exists, lists the months of each input,
and prints the columns of the optional tables (deposits, PxV, credit) so their names can be
filled into `EXTRA_OUTCOMES`. A missing required column stops the run here, before any heavy work.

In [ ]:
CELL.start("§0 · Inputs")
with step("schemas and months"):
    _req = [("customer metrics", CONFIG["MET_TABLE"], MET_REQUIRED),
            ("customer x counterparty pairs", CONFIG["PAIR_TABLE"], PAIR_REQUIRED)]
    if CONFIG["USE_HUB"]:
        _req.append(("counterparty metrics", CONFIG["CPTY_TABLE"], CPTY_REQUIRED))
    _rows, _missing, _months = [], {}, {}
    for _lab, _t, _need in _req:
        _cols = spark.table(_t).columns
        _miss = [c for c in _need if c not in _cols]
        _months[_t] = table_months(_t)
        _rows.append(dict(input=_lab, table=_t, columns=len(_cols),
                          months=f"{_months[_t][0]} .. {_months[_t][-1]} ({len(_months[_t])})",
                          missing_columns=", ".join(_miss) if _miss else "none"))
        if _miss:
            _missing[_t] = _miss
    _inputs = pd.DataFrame(_rows)

    # optional families: configured, not configured, or table not found
    _xrows, EXTRA_ACTIVE = [], []
    for _x in CONFIG["EXTRA_OUTCOMES"]:
        try:
            _dt = spark.table(_x["table"]).dtypes
        except Exception:
            _xrows.append(dict(family=_x["name"], table=_x["table"], status="table not found", columns=""))
            continue
        _names = [c for c, _ in _dt]
        _want = [_x["id_col"], _x["month_col"]] + [m[0] for m in _x["measures"].values()]
        if any(w is None for w in _want):
            _st = "not configured: fill id_col, month_col and measures in CONFIG"
        elif [w for w in _want if w not in _names]:
            _st = "configured names not in table: " + ", ".join(w for w in _want if w not in _names)
        else:
            _st = "configured"
            EXTRA_ACTIVE.append(_x)
        _xrows.append(dict(family=_x["name"], table=_x["table"], status=_st,
                           columns=", ".join(f"{c} ({t})" for c, t in _dt)))
    _extra = pd.DataFrame(_xrows, columns=["family", "table", "status", "columns"])

CELL.show(("Required inputs", _inputs), ("Optional outcome families", _extra))
if _missing:
    raise ValueError("Required columns not found, nothing was built: "
                     + "; ".join(f"{t}: {', '.join(m)}" for t, m in _missing.items()))

MONTH_MIN = CONFIG["MONTH_MIN"]
_common = sorted(set(_months[CONFIG["MET_TABLE"]]) & set(_months[CONFIG["PAIR_TABLE"]]))
_common = [m for m in _common if m >= MONTH_MIN and (CONFIG["MONTH_MAX"] is None or m <= CONFIG["MONTH_MAX"])]
if len(_common) < CONFIG["BASE_MONTHS"] + CONFIG["MIN_FOLLOW"] + 2:
    raise ValueError(f"Only {len(_common)} common months between metrics and pairs: too few for an event study.")
MONTH_MAX = _common[-1]
MAX_MI = (int(MONTH_MAX[:4]) * 12 + int(MONTH_MAX[5:7])) - _BASE
gate("months are continuous", len(_common) == MAX_MI + 1,
     f"{len(_common)} months present, {MAX_MI + 1} expected between {MONTH_MIN} and {MONTH_MAX}",
     "a missing month would look like a wave of stops")
SIG = hashlib.md5(json.dumps({k: CONFIG[k] for k in _SIG_KEYS} | {"max": MONTH_MAX},
                             sort_keys=True, default=str).encode()).hexdigest()[:10]
ans("Window", f"{MONTH_MIN} .. {MONTH_MAX} ({MAX_MI + 1} months), configuration {SIG}")

## §1 · Population

TM organisations (`is_tm`, `party_type` O, power id not shared), one row per customer per month.
**Baseline inflow** is the average monthly `amt_all_in` over the 6 months before. A customer-month is
**eligible** when it has at least one standing incoming relationship (`rec_standing_in`) and baseline
inflow above the floor. Coverage is reported as customers and as incoming dollars.

In [ ]:
CELL.start("§1 · Population")
BM = CONFIG["BASE_MONTHS"]
with step("TM organisations, baseline inflow, eligibility"):
    _met = spark.table(CONFIG["MET_TABLE"]).where(F.col("month").between(MONTH_MIN, MONTH_MAX))
    _tm = _met.where((F.col("is_tm").cast("boolean") == F.lit(True))
                     & F.col("party_type").isin(CONFIG["ORG_PARTY_TYPES"])
                     & (F.coalesce(F.col("pwr_shared").cast("boolean"), F.lit(False)) == F.lit(False)))
    _pop = _tm.select(
        F.col("mdm_id").cast("string").alias("cust"),
        F.col("customer_pwr_id").cast("string").alias("customer_pwr_id"),
        F.coalesce(F.col("master_pwr_id").cast("string"), F.col("mdm_id").cast("string")).alias("master"),
        F.col("customer_name"),
        F.coalesce(F.col("naics2").cast("string"), F.lit("NA")).alias("naics2n"),
        mi_col("month").alias("mi"),
        F.col("rec_standing_in").cast("double").alias("rec_standing_in"),
        *[F.col(c).cast("double").alias(c) for c in OUTCOMES])
    _w6 = W.partitionBy("cust").orderBy("mi").rangeBetween(-BM, -1)
    _pop = _pop.withColumn("base_in", F.sum("amt_all_in").over(_w6) / F.lit(float(BM)))
    pop = cache_parquet(_pop, "pop")

    _elig = (pop.where((F.col("mi") >= BM) & (F.col("rec_standing_in") >= 1)
                       & (F.col("base_in") >= CONFIG["BASE_IN_MIN"]))
             .withColumn("pbin", F.ntile(100).over(W.partitionBy("mi").orderBy("base_in")))
             .select("cust", "customer_pwr_id", "master", "customer_name", "naics2n", "mi", "base_in", "pbin"))
    elig = cache_parquet(_elig, "elig")

    _pm = (pop.groupBy("mi").agg(
        F.count(F.lit(1)).alias("tm_orgs"),
        F.sum((F.col("rec_standing_in") >= 1).cast("long")).alias("with_standing_in"),
        F.sum("amt_all_in").alias("in_usd"),
        F.sum(F.when(F.col("rec_standing_in") >= 1, F.col("amt_all_in")).otherwise(0.0)).alias("in_usd_standing"))
        .join(elig.groupBy("mi").agg(F.count(F.lit(1)).alias("eligible")), "mi", "left")
        .orderBy("mi").toPandas())

if _pm["tm_orgs"].sum() == 0:
    raise ValueError("No TM organisation rows found. Check is_tm and ORG_PARTY_TYPES against the "
                     "party_type values of the metrics table.")
_pm.insert(0, "month", _pm["mi"].map(mi2m))
_pm["eligible"] = _pm["eligible"].fillna(0).astype(int)
_pm["standing_share"] = _pm["with_standing_in"] / _pm["tm_orgs"]
_pm["standing_usd_share"] = _pm["in_usd_standing"] / _pm["in_usd"]
_last = _pm.iloc[-1]
_pop_sum = pd.DataFrame([dict(
    months=len(_pm), tm_orgs_last_month=int(_last.tm_orgs),
    with_standing_in_last_month=int(_last.with_standing_in),
    standing_share=float(_last.standing_share), standing_usd_share=float(_last.standing_usd_share),
    eligible_last_month=int(_last.eligible))])
CELL.show(("Population, last month", _pop_sum),
          ("By month", _pm[["month", "tm_orgs", "with_standing_in", "standing_share",
                            "standing_usd_share", "eligible"]]))
ans("TM organisations with a standing incoming relationship (last month)",
    f"{int(_last.with_standing_in):,} of {int(_last.tm_orgs):,} ({_last.standing_share:.1%}), "
    f"holding {_last.standing_usd_share:.1%} of incoming dollars")

## §2 · Stream events and customer shocks

A **stream** is customer x payer (internal or external), read from the pair table, incoming side.
A stream is **standing** going into month e when it was paid in at least 4 of the 6 months before,
including the month just before. Then, at month e:

| Type | Rule |
|---|---|
| stop | not paid in e and not in e+1 |
| skip | not paid in e, paid again in e+1 |
| short | paid in e, more than 25% below expected |
| open | not paid in e, and e is the last month of data (cannot be typed yet) |

Expected amount = median of the last 3 paid months. Lost dollars = expected (stop, skip, open) or
expected minus paid (short).

**Customer shock** = month where lost dollars are at least 10% of baseline inflow. Shock size =
lost / baseline. **Event** = first shock after 6 months without one; the first event per customer is used.
A payer is **payer-wide** when it broke at least half of its standing payees that month (5 payees or more).

In [ ]:
CELL.start("§2 · Stream events and customer shocks")
SM, STOPM, SHORT = CONFIG["STANDING_MIN"], CONFIG["STOP_MONTHS"], CONFIG["SHORT_DROP"]
TYPES = ["stop", "skip", "short", "open"]
TIER0 = CONFIG["TIER_EDGES"][0]

with step("standing streams and stream events (pair table, incoming side)"):
    _pair = (spark.table(CONFIG["PAIR_TABLE"])
             .where(F.col("month").between(MONTH_MIN, MONTH_MAX)
                    & (F.col("dir") == "in") & F.col("scope").isin("int", "ext"))
             .select(F.col("cust").cast("string").alias("cust"), "scope",
                     F.col("other").cast("string").alias("other"),
                     mi_col("month").alias("mi"), F.col("amount").cast("double").alias("amount"))
             .join(pop.select("cust").distinct(), "cust", "left_semi")
             .groupBy("cust", "scope", "other", "mi").agg(F.sum("amount").alias("amount"))
             .where(F.col("amount") > 0))
    _w = W.partitionBy("cust", "scope", "other").orderBy("mi")
    _s = (_pair
          .withColumn("cnt6", F.count(F.lit(1)).over(_w.rangeBetween(-5, 0)))
          .withColumn("lead_mi", F.lead("mi").over(_w))
          .withColumn("lead_amt", F.lead("amount").over(_w))
          .withColumn("last3", F.collect_list("amount").over(_w.rowsBetween(-2, 0)))
          .where((F.col("cnt6") >= SM) & (F.col("mi") < MAX_MI))       # standing going into e = mi + 1
          .withColumn("e", F.col("mi") + 1)
          .withColumn("expected", F.expr("element_at(array_sort(last3), cast(ceil(size(last3) / 2.0) as int))"))
          .withColumn("gap", F.col("lead_mi") - F.col("mi") - 1))
    _ev = (F.when(F.col("lead_mi") == F.col("e"),
                  F.when(F.col("lead_amt") < (1.0 - SHORT) * F.col("expected"), F.lit("short")))
           .when(F.col("lead_mi").isNull(),
                 F.when(F.lit(MAX_MI) - F.col("mi") >= STOPM, F.lit("stop")).otherwise(F.lit("open")))
           .when(F.col("gap") >= STOPM, F.lit("stop"))
           .otherwise(F.lit("skip")))
    _risk = (_s.withColumn("ev", _ev)
             .withColumn("lost", F.when(F.col("ev") == "short", F.col("expected") - F.col("lead_amt"))
                         .when(F.col("ev").isNotNull(), F.col("expected")).otherwise(F.lit(0.0)))
             .select("cust", "scope", "other", "e", "expected", "ev", "lost"))
    risk = cache_parquet(_risk, "stream_risk")

with step("payer-month roll-up and payer-wide flag"):
    _pmonth = (risk.groupBy("scope", "other", "e").agg(
        F.count(F.lit(1)).alias("n_risk"),
        F.sum(F.col("ev").isNotNull().cast("long")).alias("n_broken"),
        F.sum("lost").alias("lost"))
        .withColumn("payer_wide", ((F.col("n_risk") >= CONFIG["PAYERWIDE_MIN_PAYEES"])
                                   & (F.col("n_broken") / F.col("n_risk") >= CONFIG["PAYERWIDE_SHARE"])).cast("int")))
    payer_month = cache_parquet(_pmonth, "payer_month")

with step("stream events with payer-wide and hub flags"):
    _se = (risk.where(F.col("ev").isNotNull())
           .join(payer_month.select("scope", "other", "e", "payer_wide", "n_risk", "n_broken"),
                 ["scope", "other", "e"], "left"))
    if CONFIG["USE_HUB"]:
        _hub = (spark.table(CONFIG["CPTY_TABLE"])
                .where(F.col("month").between(MONTH_MIN, MONTH_MAX)
                       & (F.col("hub_tier").cast("int") >= CONFIG["HUB_TIER"]))
                .select(F.lit("ext").alias("scope"), F.col("ent_id").cast("string").alias("other"),
                        (mi_col("month") + 1).alias("e")).distinct().withColumn("is_hub", F.lit(1)))
        _se = _se.join(F.broadcast(_hub), ["scope", "other", "e"], "left")
    else:
        _se = _se.withColumn("is_hub", F.lit(None).cast("int"))
    stream_events = cache_parquet(_se.withColumn("is_hub", F.coalesce(F.col("is_hub"), F.lit(0))), "stream_events")

with step("customer shocks and events"):
    _lost_if = lambda cond: F.sum(F.when(cond, F.col("lost")).otherwise(0.0))
    _cs = (stream_events.groupBy("cust", "e").agg(
        F.sum("lost").alias("lost"), F.count(F.lit(1)).alias("n_streams_hit"),
        *[_lost_if(F.col("ev") == t).alias(f"lost_{t}") for t in TYPES],
        _lost_if(F.col("scope") == "int").alias("lost_int"),
        _lost_if(F.col("payer_wide") == 1).alias("lost_pw"),
        _lost_if(F.col("is_hub") == 1).alias("lost_hub"))
        .join(elig.withColumnRenamed("mi", "e"), ["cust", "e"], "inner")
        .withColumn("size", F.col("lost") / F.col("base_in"))
        .where(F.col("size") >= TIER0))
    _top = F.greatest(*[F.col(f"lost_{t}") for t in TYPES])
    _dom = F.lit("open")
    for t in reversed(TYPES):
        _dom = F.when(F.col(f"lost_{t}") == _top, F.lit(t)).otherwise(_dom)
    _wc = W.partitionBy("cust").orderBy("e")
    _sh = (_cs.withColumn("type", _dom)
           .withColumn("source", F.when(F.col("lost_int") >= 0.5 * F.col("lost"), F.lit("PNC customer payer"))
                       .otherwise(F.lit("external payer")))
           .withColumn("breadth", F.when(F.col("lost_pw") >= 0.5 * F.col("lost"), F.lit("payer-wide"))
                       .otherwise(F.lit("single-link")))
           .withColumn("hub", F.when(F.col("lost_hub") >= 0.5 * F.col("lost"), F.lit("hub payer"))
                       .otherwise(F.lit("non-hub payer")))
           .withColumn("prev_e", F.lag("e").over(_wc))
           .withColumn("is_event", (F.col("prev_e").isNull()
                                    | (F.col("e") - F.col("prev_e") > CONFIG["CLEAN_MONTHS"])).cast("int")))
    shocks = cache_parquet(_sh, "shocks")

    _e_hi = MAX_MI - CONFIG["MIN_FOLLOW"]
    _first = (shocks.where((F.col("is_event") == 1) & (F.col("e") <= _e_hi))
              .withColumn("rn", F.row_number().over(_wc)).where(F.col("rn") == 1).drop("rn", "prev_e", "is_event"))
    events = _first.toPandas()

    _tot = stream_events.groupBy("ev").agg(F.count(F.lit(1)).alias("stream_events"),
                                           F.sum("lost").alias("lost_usd")).toPandas()
    _n_risk, _n_hub = risk.count(), stream_events.where(F.col("is_hub") == 1).count()
    _n_shock_months = shocks.count()

if len(events) == 0:
    raise ValueError("No shock events found. Check the pair table values of dir / scope and the thresholds.")
_edges = CONFIG["TIER_EDGES"]
_tlabels = [f"{int(a * 100)}-{int(b * 100)}%" for a, b in zip(_edges[:-1], _edges[1:])] + [f"{int(_edges[-1] * 100)}%+"]
events["tier"] = pd.cut(events["size"], bins=_edges + [np.inf], labels=_tlabels, right=False).astype(str)
events["month"] = events["e"].map(mi2m)
_tot = _tot.set_index("ev").reindex(TYPES).fillna(0).reset_index().rename(columns={"ev": "type"})
_tot["stream_events"] = _tot["stream_events"].astype(int)
_tot["share_of_standing_stream_months"] = _tot["stream_events"] / max(_n_risk, 1)
_sum2 = pd.DataFrame([dict(standing_stream_months=_n_risk, stream_events=int(_tot.stream_events.sum()),
                           customer_shock_months=_n_shock_months, first_events=len(events),
                           event_months=f"{mi2m(BM)} .. {mi2m(_e_hi)}",
                           median_shock_size=float(events["size"].median()),
                           median_lost_usd=float(events["lost"].median()))])
CELL.show(("Streams, shocks, events", _sum2), ("Stream events by type", _tot))
if CONFIG["USE_HUB"]:
    gate("hub flag found", _n_hub > 0, f"{_n_hub:,} stream events from payers with hub_tier >= {CONFIG['HUB_TIER']}",
         "if zero, check hub_tier values; the hub cut is empty")

## §3 · Pre-flight counts, gates and headline choice

The spec's go rules, applied automatically: fewer than 1,000 events at 20% and above drops the headline
to 10%; payer-wide below 10% of events makes the headline all shocks, with payer-wide as a cut.
Figures f01 and f02 are saved here. With `RUN_STAGE = "preflight"` the notebook stops after this section.

In [ ]:
CELL.start("§3 · Pre-flight counts, gates, headline")
_by_month = (events.pivot_table(index="month", columns="type", values="cust", aggfunc="count", fill_value=0)
             .reindex(columns=[t for t in TYPES if t in set(events["type"])]).reset_index())
_by_month["all"] = _by_month.drop(columns="month").sum(axis=1)
_tier_breadth = (events.pivot_table(index="tier", columns="breadth", values="cust", aggfunc="count", fill_value=0)
                 .reindex(_tlabels).fillna(0).astype(int).reset_index())
_cuts = pd.concat([events.groupby(c).agg(events=("cust", "count"), median_size=("size", "median"),
                                         lost_usd=("lost", "sum")).reset_index()
                   .rename(columns={c: "value"}).assign(cut=c)
                   for c in ["type", "source", "breadth", "hub"]])[["cut", "value", "events", "median_size", "lost_usd"]]

_n20 = int((events["size"] >= CONFIG["HEADLINE_MIN"]).sum())
HEAD_MIN = CONFIG["HEADLINE_MIN"] if _n20 >= CONFIG["GATE_MIN_EVENTS"] else TIER0
gate(f"at least {CONFIG['GATE_MIN_EVENTS']:,} events at {CONFIG['HEADLINE_MIN']:.0%}+", _n20 >= CONFIG["GATE_MIN_EVENTS"],
     f"{_n20:,} events", f"headline threshold = {HEAD_MIN:.0%}")
_h = events[events["size"] >= HEAD_MIN]
_pw_share = float((_h["breadth"] == "payer-wide").mean()) if len(_h) else 0.0
HEAD_PW = _pw_share >= CONFIG["GATE_MIN_PAYERWIDE"]
gate(f"payer-wide is at least {CONFIG['GATE_MIN_PAYERWIDE']:.0%} of headline events", HEAD_PW,
     f"{_pw_share:.1%} of {len(_h):,}", "headline = payer-wide shocks" if HEAD_PW else "headline = all shocks, payer-wide is a cut")
HEAD_LABEL = f"{'payer-wide' if HEAD_PW else 'all'} shocks, {HEAD_MIN:.0%}+"
events["headline"] = (events["size"] >= HEAD_MIN) & ((events["breadth"] == "payer-wide") | (not HEAD_PW))
ans("Headline group", f"{HEAD_LABEL}: {int(events['headline'].sum()):,} events")

# f01: events per month by type (stacked bars)
_types_here = [t for t in TYPES if t in _by_month.columns]
fig, ax = plt.subplots(figsize=(11, 4.2))
_bottom = np.zeros(len(_by_month))
for t in _types_here:
    ax.bar(_by_month["month"], _by_month[t], bottom=_bottom, color=TYPE_COLORS[t], label=t,
           width=0.78, edgecolor=SURF, linewidth=1.2)
    _bottom += _by_month[t].to_numpy()
ax.set_title("Shock events per month, by type (first event per customer)")
ax.set_ylabel("customers"); ax.grid(axis="x", visible=False)
ax.tick_params(axis="x", rotation=60); ax.legend(ncol=len(_types_here), loc="lower right", bbox_to_anchor=(1, 1))
savefig(fig, "f01_shocks_by_month", "3", "Shock events per month by type")

# f02: shock size distribution by type
fig, ax = plt.subplots(figsize=(8.5, 4.2))
_bins = np.linspace(TIER0, 1.5, 29)
ax.hist([events.loc[events["type"] == t, "size"].clip(upper=1.5) for t in _types_here], bins=_bins,
        stacked=True, color=[TYPE_COLORS[t] for t in _types_here], label=_types_here, edgecolor=SURF, linewidth=0.8)
for _x in _edges[1:]:
    ax.axvline(_x, color=INK2, linewidth=1, linestyle=":")
ax.set_title("Shock size: lost dollars / baseline monthly inflow (capped at 150%)")
ax.set_xlabel("shock size"); ax.set_ylabel("customers"); ax.legend(ncol=len(_types_here), loc="lower right", bbox_to_anchor=(1, 1))
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
savefig(fig, "f02_shock_size", "3", "Distribution of shock size by type; dotted lines are the tier edges")

CELL.show(("Gates", gates_frame()), ("Events by tier and breadth", _tier_breadth),
          ("Events by cut", _cuts), ("Events by month and type", _by_month))
RUN_FULL = CONFIG["RUN_STAGE"] == "full"
if not RUN_FULL:
    CELL.log("RUN_STAGE = preflight: stopping here. Set RUN_STAGE = 'full' to continue (work files are reused).")

## §4 · Matched controls and the event panel

Up to 3 controls per shocked customer: same event month, same NAICS2, same baseline-inflow decile, nearest
on baseline inflow, a different master, and no shock in the 6 months before or in the event month.
Controls are **not** required to stay shock-free afterwards (`CONTROL_CLEAN_AFTER = 0`): picking controls
on their future would make them look healthier than any real customer and overstate every effect.
The panel holds every unit for months -6 to +6. Months past the end of data stay empty (censored);
months inside the data with no metrics row count as zero activity.

In [ ]:
if RUN_FULL:
    CELL.start("§4 · Matched controls and the event panel")
    PRE, POST, NC = CONFIG["PRE_MONTHS"], CONFIG["POST_MONTHS"], CONFIG["N_CONTROLS"]
    # deterministic cap: headline events first, then by a stable hash of the customer id
    events["_h"] = events["cust"].map(lambda s: int(hashlib.md5(str(s).encode()).hexdigest()[:8], 16))
    _pe = (events.sort_values(["headline", "_h"], ascending=[False, True])
           .head(CONFIG["PANEL_MAX_EVENTS"]).reset_index(drop=True))
    with step("matching controls"):
        _evs = spark.createDataFrame(_pe[["cust", "master", "e", "naics2n", "pbin", "base_in"]]
                                     .rename(columns={"cust": "ev", "master": "ev_master", "base_in": "ev_base"})
                                     .astype({"e": "int32", "pbin": "int32"}))
        _arr = shocks.groupBy("cust").agg(F.collect_set("e").alias("shock_arr"))
        _cand = (elig.select(F.col("cust").alias("unit"), F.col("master").alias("u_master"),
                             F.col("mi").alias("e"), "naics2n", "pbin", F.col("base_in").alias("ubase"))
                 .join(_arr.withColumnRenamed("cust", "unit"), "unit", "left")
                 .withColumn("shock_arr", F.coalesce(F.col("shock_arr"), F.array().cast("array<int>"))))
        _dec = lambda c: F.ceil(F.col(c) / 10.0).cast("int")          # percentile bin -> decile
        _m = (_evs.withColumn("dec", _dec("pbin")).drop("pbin")
              .join(_cand.withColumn("dec", _dec("pbin")).drop("pbin"), ["e", "naics2n", "dec"], "inner")
              .where((F.col("unit") != F.col("ev")) & (F.col("u_master") != F.col("ev_master"))
                     & F.expr(f"NOT exists(shock_arr, x -> x - e >= -{CONFIG['CLEAN_MONTHS']} AND x - e <= {CONFIG['CONTROL_CLEAN_AFTER']})"))
              .withColumn("dist", F.abs(F.log(F.col("ubase")) - F.log(F.col("ev_base"))))
              .withColumn("rn", F.row_number().over(
                  W.partitionBy("ev").orderBy("dist", F.xxhash64("unit", "ev"))))
              .where(F.col("rn") <= NC)
              .select("ev", "e", "unit", "ubase", F.lit("C").alias("role")))
        _units = (_evs.select("ev", "e", F.col("ev").alias("unit"), F.col("ev_base").alias("ubase"),
                              F.lit("S").alias("role")).unionByName(_m))
        units = cache_parquet(_units, f"units_{len(_pe)}_{NC}_{CONFIG['CONTROL_CLEAN_AFTER']}_{int(HEAD_MIN * 100)}_{int(HEAD_PW)}")

    with step("event panel (months -6 .. +6) from the customer metrics"):
        _grid = (units.withColumn("t", F.explode(F.sequence(F.lit(-PRE), F.lit(POST))))
                 .withColumn("mi", F.col("e") + F.col("t"))
                 .where((F.col("mi") >= 0) & (F.col("mi") <= MAX_MI)))
        _pan = _grid.join(pop.select(F.col("cust").alias("unit"), "mi", "customer_pwr_id", *OUTCOMES),
                          ["unit", "mi"], "left")
        EXTRA_COLS = {}
        if EXTRA_ACTIVE:
            _ids = (pop.select(F.col("cust").alias("unit"), "customer_pwr_id").where(F.col("customer_pwr_id").isNotNull())
                    .dropDuplicates(["unit"]).join(units.select("unit").distinct(), "unit", "left_semi"))
            for _x in EXTRA_ACTIVE:
                _key = "unit" if _x["id_kind"] == "mdm_id" else "customer_pwr_id"
                _mc = F.col(_x["month_col"])
                _mstr = F.when(_mc.cast("string").rlike(r"^\d{4}-\d{2}$"), _mc.cast("string")) \
                         .otherwise(F.date_format(_mc.cast("date"), "yyyy-MM"))
                _xt = (spark.table(_x["table"])
                       .select(F.col(_x["id_col"]).cast("string").alias(_key), mi_col(_mstr).alias("mi"),
                               *[F.col(src).cast("double").alias(out) for out, (src, _k) in _x["measures"].items()])
                       .groupBy(_key, "mi").agg(*[F.avg(out).alias(out) for out in _x["measures"]]))
                if _key == "customer_pwr_id":
                    _xt = _xt.join(_ids, "customer_pwr_id", "inner").drop("customer_pwr_id")
                _pan = _pan.join(_xt, ["unit", "mi"], "left")
                for out, (_src, _k) in _x["measures"].items():
                    EXTRA_COLS[out] = (f"{_x['name']}: {out}", _k)
        panel = _pan.drop("customer_pwr_id").toPandas()

    _nctl = panel[panel.role == "C"].groupby("ev")["unit"].nunique().reindex(_pe["cust"]).fillna(0).astype(int)
    _pe["n_controls"] = _nctl.to_numpy()
    _match = (_pe.groupby("n_controls").agg(events=("cust", "count")).reset_index())
    _pe = _pe[_pe["n_controls"] > 0].reset_index(drop=True)
    _ctl_base = panel[(panel.role == "C")].drop_duplicates(["ev", "unit"]).groupby("ev")["ubase"].mean()
    _bal = pd.DataFrame([dict(
        events_in_panel=len(_pe), control_units=int(panel[panel.role == "C"][["ev", "unit"]].drop_duplicates().shape[0]),
        panel_rows=len(panel),
        median_base_in_shocked=float(_pe["base_in"].median()),
        median_base_in_controls=float(_ctl_base.reindex(_pe["cust"]).median()))])
    gate("events with at least one control", len(_pe) >= 0.8 * max(int(_match.events.sum()), 1),
         f"{len(_pe):,} of {int(_match.events.sum()):,}", "unmatched events are left out of the effects")
    CELL.show(("Panel", _bal), ("Controls found per event", _match))

## §5 · Effects

For each outcome and each month, every unit is measured against its own baseline (months -6 to -1). The
effect is the shocked group's value minus the matched control group's value.

- **index** outcomes (dollars, counts): 100 x value / baseline; group value = **median**; effect in index points.
- **diff** outcomes (shares, flags): change from baseline; group value = **mean** (shares in percentage points).
- Intervals: bootstrap, resampled by master so one family counts once.
- **Pre-trend check**: baseline moved to months -6..-4, effect read at -3..-1. It should be near zero;
  if it is not, shocked customers were already drifting before the shock.

Figures f03 to f05, f10, f11, f15 are saved; the full effect table goes to `p1_effects.csv`.

In [ ]:
if RUN_FULL:
    CELL.start("§5 · Effects")
    ALL_OUT = dict(OUTCOMES); ALL_OUT.update(EXTRA_COLS)
    TS = np.arange(-PRE, POST + 1)
    ET, B = CONFIG["EFFECT_T"], CONFIG["BOOT_REPS"]
    _rng = np.random.default_rng(CONFIG["SEED"])
    _evmeta = _pe.set_index("cust")

    def _matrix(col, kind, base_ts):
        """Units x months matrix of the outcome relative to the unit's own baseline."""
        p = panel[["ev", "role", "unit", "t", "ubase", col]].copy()
        if col in OUTCOMES and (kind in ("index", "scaled") or col == "rec_broken_out"):
            p[col] = p[col].fillna(0.0)               # inside the data, no metrics row = no activity
        if kind == "scaled":
            p[col] = (p[col] / p["ubase"]).clip(-5, 5) * 100.0
        if col in SHARE_COLS:
            p[col] = p[col] * 100.0
        m = p.set_index(["ev", "role", "unit", "t"])[col].unstack("t").reindex(columns=TS)
        base = m[base_ts].mean(axis=1, skipna=True)
        if kind == "index":
            m = m.div(base.where(base > 0), axis=0).mul(100.0).clip(0, CONFIG["INDEX_CAP"])
        else:
            m = m.sub(base, axis=0)
        return m

    def _sc(col, kind, base_ts=None):
        """Shocked rows (one per event) and control rows (one per control unit, labelled by its event)."""
        m = _matrix(col, kind, list(range(-PRE, 0)) if base_ts is None else base_ts)
        S = m.xs("S", level="role").droplevel("unit")
        C = m.xs("C", level="role").droplevel("unit")
        S = S.loc[S.index.intersection(C.index.unique())]
        return S, C

    def _stat(S, C, fn):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return fn(S, axis=0), fn(C, axis=0)

    def _boot(S, C, cl_s, cl_c, fn):
        """Cluster bootstrap by master: a drawn master brings its events and their controls."""
        uniq, codes_s = np.unique(cl_s, return_inverse=True)
        codes_c = np.searchsorted(uniq, cl_c)
        rs, rc = np.arange(len(codes_s)), np.arange(len(codes_c))
        out = np.full((B, S.shape[1]), np.nan)
        for b in range(B):
            draws = np.bincount(_rng.integers(0, len(uniq), len(uniq)), minlength=len(uniq))
            a, c = _stat(S[np.repeat(rs, draws[codes_s])], C[np.repeat(rc, draws[codes_c])], fn)
            out[b] = a - c
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return np.nanpercentile(out, 2.5, axis=0), np.nanpercentile(out, 97.5, axis=0)

    def _effect(S, C, ids, fn):
        """Group statistic of the shocked minus the same statistic of their controls, with interval."""
        ii = S.index.intersection(pd.Index(ids))
        if len(ii) < 20:
            return None
        Cg = C.loc[C.index.isin(ii)]
        s, c = S.loc[ii].to_numpy(float), Cg.to_numpy(float)
        ms, mc = _stat(s, c, fn)
        lo, hi = _boot(s, c, _evmeta.loc[ii, "master"].to_numpy().astype(str),
                       _evmeta.loc[Cg.index, "master"].to_numpy().astype(str), fn)
        return ms, mc, ms - mc, lo, hi, np.sum(~np.isnan(s), axis=0)

    GROUPS = {"headline": _pe.loc[_pe["headline"], "cust"], "all shocks 10%+": _pe["cust"]}
    for _c, _pref in [("tier", "tier "), ("type", "type "), ("source", ""), ("breadth", ""), ("hub", "")]:
        for _v in sorted(_pe[_c].unique()):
            GROUPS[f"{_pref}{_v}"] = _pe.loc[_pe[_c] == _v, "cust"]
    GROUP_CUT = {"headline": "headline", "all shocks 10%+": "headline"}
    GROUP_CUT.update({g: ("tier" if g.startswith("tier ") else "type" if g.startswith("type ") else "cut")
                      for g in GROUPS if g not in GROUP_CUT})

    _rows, _pre_rows = [], []
    with step(f"effects: {len(ALL_OUT)} outcomes x {len(GROUPS)} groups, {B} bootstrap draws"):
        for col, (label, kind) in ALL_OUT.items():
            fn = np.nanmedian if kind == "index" else np.nanmean
            S, C = _sc(col, kind)
            for g, ids in GROUPS.items():
                r = _effect(S, C, ids, fn)
                if r is None:
                    continue
                ms, mc, md, lo, hi, nobs = r
                for j, t in enumerate(TS):
                    _rows.append(dict(group=g, cut=GROUP_CUT[g], outcome=col, label=label, kind=kind, t=int(t),
                                      n=int(nobs[j]), shocked=ms[j], control=mc[j], effect=md[j], lo=lo[j], hi=hi[j]))
            # pre-trend check on the headline group
            Sp, Cp = _sc(col, kind, base_ts=list(range(-PRE, -3)))
            r = _effect(Sp, Cp, GROUPS["headline"], fn)
            if r is not None:
                ms, mc, md, lo, hi, nobs = r
                for t in (-3, -2, -1):
                    j = int(np.where(TS == t)[0][0])
                    _pre_rows.append(dict(outcome=col, label=label, kind=kind, t=t, n=int(nobs[j]),
                                          effect=md[j], lo=lo[j], hi=hi[j]))
    effects = pd.DataFrame(_rows)
    pretrend = pd.DataFrame(_pre_rows)
    effects.to_csv(os.path.join(OUT_DIR, "p1_effects.csv"), index=False)
    pretrend.to_csv(os.path.join(OUT_DIR, "p1_pretrend.csv"), index=False)
    _pe.drop(columns=["_h"]).to_parquet(os.path.join(OUT_DIR, "p1_events.parquet"), index=False)

    _unit = lambda kind: "index points" if kind == "index" else "pp" if kind == "scaled" else "change"

    def _event_fig(cols, name, title):
        """Left: shocked and controls. Right: the difference with its interval. One row per outcome."""
        e = effects[(effects.group == "headline") & effects.outcome.isin(cols)]
        if e.empty:
            return
        fig, axes = plt.subplots(len(cols), 2, figsize=(11, 3.4 * len(cols)), squeeze=False)
        for r, col in enumerate(cols):
            x = e[e.outcome == col].sort_values("t")
            if x.empty:
                continue
            a, b = axes[r]
            a.plot(x.t, x.control, color=C_CTRL, marker="o", markersize=4, label="matched controls")
            a.plot(x.t, x.shocked, color=C_SHOCK, marker="o", markersize=4, label="shocked")
            a.axvline(0, color=INK2, linewidth=1, linestyle=":")
            a.set_title(x.label.iloc[0]); a.set_ylabel("index, baseline = 100" if x.kind.iloc[0] == "index" else "change from baseline")
            a.legend(loc="best")
            b.fill_between(x.t, x.lo, x.hi, color=C_SHOCK, alpha=0.18, linewidth=0)
            b.plot(x.t, x.effect, color=C_SHOCK, marker="o", markersize=4)
            b.axhline(0, color=INK2, linewidth=1); b.axvline(0, color=INK2, linewidth=1, linestyle=":")
            b.set_title("Shocked minus controls (95% interval)"); b.set_ylabel(_unit(x.kind.iloc[0]))
            for ax_ in (a, b):
                ax_.set_xticks(TS)
            if r == len(cols) - 1:
                a.set_xlabel("months from the shock"); b.set_xlabel("months from the shock")
        fig.suptitle(f"{title} — {HEAD_LABEL}, n = {int(e.n.max()):,}", x=0.01, ha="left", fontsize=12, fontweight="bold")
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        savefig(fig, name, "5", f"{title}: shocked against matched controls, months -{PRE} to +{POST}")
        try:
            import plotly.graph_objects as go
            from plotly.subplots import make_subplots
            pf = make_subplots(rows=len(cols), cols=2, subplot_titles=sum(
                [[e[e.outcome == c].label.iloc[0], "Shocked minus controls"] for c in cols if not e[e.outcome == c].empty], []))
            for r, col in enumerate(cols, start=1):
                x = e[e.outcome == col].sort_values("t")
                if x.empty:
                    continue
                pf.add_trace(go.Scatter(x=x.t, y=x.control, name="matched controls", mode="lines+markers",
                                        line=dict(color=C_CTRL), showlegend=(r == 1)), row=r, col=1)
                pf.add_trace(go.Scatter(x=x.t, y=x.shocked, name="shocked", mode="lines+markers",
                                        line=dict(color=C_SHOCK), showlegend=(r == 1)), row=r, col=1)
                pf.add_trace(go.Scatter(x=list(x.t) + list(x.t[::-1]), y=list(x.hi) + list(x.lo[::-1]), fill="toself",
                                        fillcolor="rgba(42,120,214,0.18)", line=dict(width=0), hoverinfo="skip",
                                        showlegend=False), row=r, col=2)
                pf.add_trace(go.Scatter(x=x.t, y=x.effect, name="difference", mode="lines+markers",
                                        line=dict(color=C_SHOCK), showlegend=False,
                                        customdata=np.stack([x.lo, x.hi, x.n], axis=1),
                                        hovertemplate="month %{x}: %{y:.1f} (%{customdata[0]:.1f} to %{customdata[1]:.1f}), n=%{customdata[2]:,}<extra></extra>"),
                             row=r, col=2)
            pf.update_layout(title=f"{title} — {HEAD_LABEL}", template="simple_white", hovermode="x unified",
                             height=340 * len(cols), paper_bgcolor=SURF, plot_bgcolor=SURF)
            save_plotly(pf, name, "5", title)
        except ImportError:
            pass

    _key = ["amt_all_in", "amt_all_out", "ntxn_all_out", "ncpty_ext_out", "rec_broken_share_out",
            "rec_broken_amt_share_out", "net_flow_all"] + list(EXTRA_COLS)
    with step("figures f03 to f05, f10, f11, f15"):
        _event_fig(["amt_all_in"], "f03_inflow_event_time", "Incoming dollars around the shock")
        _event_fig(["amt_all_out", "ntxn_all_out"], "f04_outflow_event_time", "Own outgoing payments around the shock")
        _event_fig(["rec_broken_share_out", "rec_broken_amt_share_out"], "f05_own_broken_share",
                   "Own standing payees not paid around the shock")
        _xcols = list(EXTRA_COLS)
        if _xcols:
            _event_fig(_xcols, "f06_extra_event_time", "Deposits, PxV and credit around the shock")

        def _dot_fig(sub, groups, name, title, caption):
            cols = [c for c in _key if c in set(sub.outcome)]
            if not cols or not groups:
                return
            ncol = 3; nrow = int(math.ceil(len(cols) / ncol))
            fig, axes = plt.subplots(nrow, ncol, figsize=(13, 0.9 + (0.42 * len(groups) + 1.5) * nrow), squeeze=False)
            for k, col in enumerate(cols):
                ax = axes[k // ncol][k % ncol]
                x = sub[sub.outcome == col].set_index("group").reindex(groups)
                y = np.arange(len(groups))[::-1]
                ax.hlines(y, x.lo, x.hi, color=C_SHOCK, linewidth=2, alpha=0.55)
                ax.plot(x.effect, y, "o", color=C_SHOCK, markersize=7, markeredgecolor=SURF, markeredgewidth=1.5)
                ax.axvline(0, color=INK2, linewidth=1)
                ax.set_yticks(y); ax.set_yticklabels(groups if k % ncol == 0 else [])
                ax.set_title(x.label.dropna().iloc[0] if x.label.notna().any() else col, fontsize=9.5)
                ax.grid(axis="y", visible=False)
            for k in range(len(cols), nrow * ncol):
                axes[k // ncol][k % ncol].axis("off")
            fig.suptitle(title, x=0.01, ha="left", fontsize=12, fontweight="bold")
            fig.tight_layout(rect=(0, 0, 1, 0.96))
            savefig(fig, name, "5", caption)

        _at = effects[effects.t == ET]
        _dot_fig(_at[_at.cut == "tier"], [g for g in [f"tier {l}" for l in _tlabels] if g in set(_at.group)],
                 "f10_dose_response", f"Dose response: effect at month +{ET} by shock size (shocked minus controls, 95% interval)",
                 f"Effect at month +{ET} by shock tier, one panel per outcome")
        _dot_fig(_at[_at.cut.isin(["type", "cut"])], [g for g in GROUPS if GROUP_CUT[g] in ("type", "cut") and g in set(_at.group)],
                 "f11_cuts", f"Effect at month +{ET} by shock type, source, breadth and hub (95% interval)",
                 f"Effect at month +{ET} by cut, one panel per outcome")
        if not pretrend.empty:
            _pt = pretrend.assign(group=pretrend.t.map(lambda t: f"month {t}"))
            _dot_fig(_pt, ["month -3", "month -2", "month -1"], "f15_placebo",
                     "Pre-trend check: effect before the shock, baseline moved to months -6..-4 (should sit on zero)",
                     "Pre-trend (placebo) check on the headline group")

    _hl = effects[(effects.group == "headline") & effects.outcome.isin(_key)]
    _wide = (_hl[_hl.t.isin([-1, 0, 1, 3, 6])].pivot_table(index=["label", "kind"], columns="t", values="effect")
             .rename(columns=lambda t: f"effect t={t:+d}").reset_index())
    _at3 = _hl[_hl.t == ET][["label", "n", "shocked", "control", "effect", "lo", "hi"]].copy()
    _at3["clear"] = (_at3.lo > 0) | (_at3.hi < 0)
    _dose = (effects[(effects.cut == "tier") & (effects.t == ET) & effects.outcome.isin(_key)]
             .pivot_table(index="label", columns="group", values="effect").reset_index())
    _ptshow = pretrend[pretrend.t == -1][["label", "effect", "lo", "hi"]].copy() if not pretrend.empty else pd.DataFrame()
    if len(_ptshow):
        _ptshow["flat"] = (_ptshow.lo <= 0) & (_ptshow.hi >= 0)
        gate("pre-trend is flat for outgoing dollars",
             bool(_ptshow.loc[_ptshow.label == OUTCOMES["amt_all_out"][0], "flat"].all()),
             "effect at month -1 with baseline -6..-4", "if it fails, read effects as shock plus an earlier drift")
    CELL.show((f"Headline ({HEAD_LABEL}): effect at month +{ET}", _at3),
              ("Headline: effect by month", _wide),
              (f"Dose response: effect at month +{ET} by shock size", _dose),
              ("Pre-trend check at month -1 (should include zero)", _ptshow))
    for _, r in _at3.iterrows():
        ans(f"Effect at +{ET}: {r.label}", f"{r.effect:+.1f} ({r.lo:+.1f} to {r.hi:+.1f})"
            + ("" if r.clear else " — interval includes zero"))

## §6 · Second hop

Does the shock pass on? For every shocked customer and control that pays other PNC customers through
standing streams: the share of those standing streams that break (stop, skip, short), month by month.
Read from the receivers' side, so this is the next customer's incoming shock.

In [ ]:
if RUN_FULL:
    CELL.start("§6 · Second hop")
    with step("standing internal streams paid by shocked customers and controls"):
        _u = units.select("ev", "e", "unit", "role").join(
            spark.createDataFrame(_pe[["cust", "headline"]].rename(columns={"cust": "ev"})), "ev", "inner")
        _hop = (payer_month.where(F.col("scope") == "int")
                .select(F.col("other").alias("unit"), F.col("e").alias("e2"), "n_risk", "n_broken")
                .join(F.broadcast(_u), "unit", "inner")
                .withColumn("t", F.col("e2") - F.col("e"))
                .where(F.col("t").between(-PRE, POST))
                .groupBy("headline", "role", "t").agg(F.countDistinct("unit").alias("payers"),
                                                       F.sum("n_risk").alias("streams"),
                                                       F.sum("n_broken").alias("broken"))
                .toPandas())
    if _hop.empty or _hop[_hop.role == "S"].streams.sum() == 0:
        CELL.show(("Second hop", pd.DataFrame([dict(note="no shocked customer pays other PNC customers through standing streams")])))
    else:
        def _hop_tab(d):
            d = d.groupby(["role", "t"])[["payers", "streams", "broken"]].sum().reset_index()
            d["broken_share"] = d.broken / d.streams
            w = d.pivot(index="t", columns="role", values=["broken_share", "streams", "payers"])
            w.columns = [f"{a}_{'shocked' if b == 'S' else 'controls'}" for a, b in w.columns]
            return w.reset_index()
        _hh = _hop_tab(_hop[_hop.headline]); _ha = _hop_tab(_hop)
        _use, _lab = (_hh, HEAD_LABEL) if "broken_share_shocked" in _hh and _hh.get("streams_shocked", pd.Series([0])).sum() >= 200 else (_ha, "all shocks 10%+")
        fig, ax = plt.subplots(figsize=(8.5, 4.2))
        if "broken_share_controls" in _use:
            ax.plot(_use.t, 100 * _use.broken_share_controls, color=C_CTRL, marker="o", markersize=4, label="payees of matched controls")
        ax.plot(_use.t, 100 * _use.broken_share_shocked, color=C_SHOCK, marker="o", markersize=4, label="payees of shocked customers")
        ax.axvline(0, color=INK2, linewidth=1, linestyle=":")
        ax.set_title(f"Second hop: standing streams to other PNC customers that break — {_lab}")
        ax.set_xlabel("months from the payer's shock"); ax.set_ylabel("% of standing streams broken"); ax.set_xticks(TS); ax.legend()
        savefig(fig, "f14_second_hop", "6", "Share of the shocked customers' own standing internal payment streams that break, against controls")
        _pre_m = _use[_use.t.between(-3, -1)]; _post_m = _use[_use.t.between(1, 3)]
        _rate = lambda d, k: float(d[f"broken_share_{k}"].mul(d[f"streams_{k}"]).sum() / max(d[f"streams_{k}"].sum(), 1)) if f"streams_{k}" in d else float("nan")
        _h2 = pd.DataFrame([dict(group=_lab,
                                 shocked_before=_rate(_pre_m, "shocked"), shocked_after=_rate(_post_m, "shocked"),
                                 controls_before=_rate(_pre_m, "controls"), controls_after=_rate(_post_m, "controls"))])
        _h2["extra_break_pp"] = 100 * ((_h2.shocked_after - _h2.shocked_before) - (_h2.controls_after - _h2.controls_before))
        CELL.show(("Broken share of standing internal streams: months -3..-1 against +1..+3", _h2),
                  (f"By month ({_lab})", _use))
        ans("Second hop: extra break rate on streams to other PNC customers, months +1..+3",
            f"{float(_h2.extra_break_pp.iloc[0]):+.2f} pp ({_lab})")

## §7 · Case panels

The largest shocks by lost dollars, plus named cases, one figure each: monthly inflow as green bars above
zero, outflow as red bars below, the shock month marked. The table lists the streams that broke.

In [ ]:
if RUN_FULL:
    CELL.start("§7 · Case panels")
    with step("case selection and series"):
        _cases = events.sort_values("lost", ascending=False).head(CONFIG["CASE_TOP_N"])[["cust", "customer_name", "e", "lost", "size", "type", "breadth"]]
        for _p in CONFIG["CASE_NAME_PATTERNS"]:
            _nm = (pop.where(F.upper(F.col("customer_name")).contains(_p.upper()))
                   .groupBy("cust", "customer_name").agg(F.sum("amt_all_in").alias("tot")).orderBy(F.desc("tot")).limit(1).toPandas())
            if len(_nm) and _nm.cust.iloc[0] not in set(_cases.cust):
                _sh1 = shocks.where(F.col("cust") == _nm.cust.iloc[0]).orderBy(F.desc("lost")).limit(1).toPandas()
                _cases = pd.concat([_cases, pd.DataFrame([dict(
                    cust=_nm.cust.iloc[0], customer_name=_nm.customer_name.iloc[0],
                    e=int(_sh1.e.iloc[0]) if len(_sh1) else -1, lost=float(_sh1.lost.iloc[0]) if len(_sh1) else np.nan,
                    size=float(_sh1["size"].iloc[0]) if len(_sh1) else np.nan,
                    type=_sh1["type"].iloc[0] if len(_sh1) else "no shock found",
                    breadth=_sh1.breadth.iloc[0] if len(_sh1) else "")])], ignore_index=True)
        _ids = list(_cases.cust)
        _ser = pop.where(F.col("cust").isin(_ids)).select("cust", "mi", "amt_all_in", "amt_all_out").toPandas()
        _keys = spark.createDataFrame(_cases[_cases.e >= 0][["cust", "e"]].astype({"e": "int32"}))
        _brk = (stream_events.join(F.broadcast(_keys), ["cust", "e"], "inner")
                .withColumn("rn", F.row_number().over(W.partitionBy("cust").orderBy(F.desc("lost"))))
                .where(F.col("rn") <= 5).select("cust", "scope", "other", "ev", "expected", "lost", "payer_wide", "n_risk", "n_broken")
                .toPandas())
    for k, c in enumerate(_cases.itertuples(), start=1):
        s = _ser[_ser.cust == c.cust].set_index("mi").reindex(range(MAX_MI + 1)).fillna(0.0)
        fig, ax = plt.subplots(figsize=(11, 4))
        x = np.arange(MAX_MI + 1)
        ax.bar(x, s.amt_all_in / 1e6, color=C_IN, width=0.78, label="inflow")
        ax.bar(x, -s.amt_all_out / 1e6, color=C_OUT, width=0.78, label="outflow")
        ax.axhline(0, color=INK2, linewidth=1)
        if c.e >= 0:
            ax.axvline(c.e, color=INK, linewidth=1.2, linestyle="--")
            ax.annotate(f"shock {mi2m(c.e)}: {usd(c.lost)} lost ({c.size:.0%} of baseline), {c.type}",
                        xy=(c.e, ax.get_ylim()[1]), xytext=(4, -12), textcoords="offset points", fontsize=9, color=INK)
        ax.set_xticks(x[::2]); ax.set_xticklabels([mi2m(i) for i in x[::2]], rotation=60)
        ax.set_ylabel("USD millions per month"); ax.grid(axis="x", visible=False); ax.legend(loc="lower right", bbox_to_anchor=(1, 1), ncol=2)
        ax.set_title(f"Case {k}: {c.customer_name}")
        savefig(fig, f"f16_case_{k:02d}", "7", f"Case panel: {c.customer_name}, inflow (green) and outflow (red) by month")
    _cases_show = _cases.assign(month=_cases.e.map(lambda i: mi2m(i) if i >= 0 else "")).drop(columns=["e"])
    _brk = _brk.merge(_cases[["cust", "customer_name"]], on="cust").drop(columns=["cust"])
    CELL.show(("Cases", _cases_show[["customer_name", "month", "lost", "size", "type", "breadth"]]),
              ("Largest broken streams of each case (other = payer id; n_risk / n_broken = that payer's standing payees and how many broke)",
               _brk[["customer_name", "scope", "other", "ev", "expected", "lost", "payer_wide", "n_risk", "n_broken"]]))

## §8 · Answer sheet, gates and files

One output to screenshot at the end of a run.

In [ ]:
CELL.start("§8 · Answer sheet, gates and files")
pd.DataFrame(FIG_INDEX).to_csv(os.path.join(FIG_DIR, "figures_index.csv"), index=False)
gates_frame().to_csv(os.path.join(OUT_DIR, "p1_gates.csv"), index=False)
pd.DataFrame(ANSWERS).to_csv(os.path.join(OUT_DIR, "p1_answers.csv"), index=False)
_files = pd.DataFrame(FIG_INDEX, columns=["file", "section", "caption"])
CELL.show(("Answers", pd.DataFrame(ANSWERS)), ("Gates", gates_frame()),
          (f"Figures saved in {FIG_DIR}", _files),
          ("Timings (minutes)", pd.DataFrame(TIMINGS, columns=["step", "minutes"])))